# ⚡ Notebook 12: Flash, Paged, and Ring Attention

Advanced attention mechanisms for long sequences: **Flash Attention** (tiled, O(n) memory), **Paged Attention** (virtual memory for KV-cache), and **Ring Attention** (distributed sequence parallelism).

This notebook derives each algorithm from first principles, implements them in MLX, and benchmarks them against standard attention.

**Prerequisites:** Notebooks 01–11

**What you'll learn:**
- 💡 Why standard attention is O(n²) and impractical for long sequences
- 🎯 The online softmax algorithm — the mathematical core of Flash Attention
- ⚡ Tiled Flash Attention — exact attention without the O(n²) memory cost
- 💡 Paged Attention — virtual memory for KV-cache (vLLM)
- 🎯 Ring Attention — distributing attention across devices
- ⚡ Benchmarks comparing standard vs flash attention at various sequence lengths

In [ ]:
from utils.checks import validate_environment, print_environment_report
env = print_environment_report()

## The O(n²) Problem

⚠️ Standard attention computes the full (seq_len × seq_len) attention matrix. For long sequences, this is catastrophic:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

The $QK^T$ product creates an $n \times n$ matrix. For seq_len=128K, that's $128K^2 \times 2$ bytes = **32 GB** just for attention weights in float16!

💡 Flash Attention solves this by computing attention in **tiles**, never materializing the full matrix.

In [ ]:
import mlx.core as mx
import numpy as np

# 💡 Memory analysis: why standard attention breaks at long sequences
print("Standard attention memory (float16):")
print(f"{'seq_len':>10s}  {'attn matrix':>12s}  {'status':>10s}")
print("-" * 38)
for seq_len in [1024, 4096, 16384, 65536, 131072]:
    mem_gb = seq_len**2 * 2 / 1e9  # float16
    status = "✅ OK" if mem_gb < 8 else "⚠️ tight" if mem_gb < 20 else "❌ OOM"
    print(f"{seq_len:>10d}  {mem_gb:>10.2f} GB  {status:>10s}")
print(f"\n⚠️ Standard attention is impractical beyond ~16K tokens on most hardware!")

## Online Softmax — The Core Algorithm

🎯 **Interview tip:** Online softmax is the mathematical foundation of Flash Attention. Understanding it is essential.

### The Problem

Standard softmax requires **two passes** over the data:
1. Find the maximum: $m = \max_i(x_i)$
2. Compute: $\text{softmax}(x_i) = \frac{e^{x_i - m}}{\sum_j e^{x_j - m}}$

This means we need to see **all** values before computing any output — impossible if we're processing blocks at a time.

### The Solution: Online (Streaming) Softmax

💡 We can compute softmax **incrementally** using running statistics:

**State:** running max $m$ and running sum $\ell = \sum e^{x_i - m}$

**Update rule** (when processing a new block of values $x_{\text{new}}$):

$$m_{\text{new}} = \max(m, \max(x_{\text{new}}))$$
$$\ell_{\text{new}} = \ell \cdot e^{m - m_{\text{new}}} + \sum e^{x_{\text{new}} - m_{\text{new}}}$$

**Final result:** $\text{softmax}(x_i) = \frac{e^{x_i - m}}{\ell}$

⚡ The key insight: when the max changes, we **rescale** the previous sum. This is exact — no approximation!

In [ ]:
from utils.attention_optimization import online_softmax, online_softmax_blocked

# 💡 Demo: online softmax matches standard softmax exactly
mx.random.seed(42)
x = mx.random.normal((4, 16))
mx.eval(x)

standard = mx.softmax(x, axis=-1)
online = online_softmax(x)
online_blk = online_softmax_blocked(x, block_size=4)
mx.eval(standard, online, online_blk)

print("Online softmax vs standard softmax:")
print(f"  Element-by-element max diff: {float(mx.max(mx.abs(standard - online))):.2e}")
print(f"  Blocked (bs=4) max diff:     {float(mx.max(mx.abs(standard - online_blk))):.2e}")
print(f"\n💡 Both variants produce identical results — the online algorithm is exact!")

# Show the values match
print(f"\nFirst row comparison:")
print(f"  Standard: {np.array(standard[0, :4]).round(4)}")
print(f"  Online:   {np.array(online[0, :4]).round(4)}")

## Flash Attention — Tiled Attention Without O(n²) Memory

🎯 **Interview tip:** Flash Attention (Dao et al., 2022) is one of the most impactful systems papers in modern ML. It computes **exact** attention without materializing the full $n \times n$ matrix.

### Algorithm

Flash Attention tiles the computation into blocks:

```
For each Q block i:
    For each KV block j:
        1. S_ij = Q_i @ K_j^T / √d     (block_size × block_size tile)
        2. Online softmax update (m, ℓ)  (running max and sum)
        3. O_i += softmax_tile @ V_j     (rescaled accumulation)
    Normalize O_i by final ℓ
```

### Memory Analysis

| | Standard | Flash |
|---|---|---|
| **Peak memory** | $O(n^2 + n \cdot d)$ | $O(n \cdot d + B^2)$ |
| **HBM reads** | $O(n^2 \cdot d)$ | $O(n^2 \cdot d / B)$ |
| **Exact?** | Yes | Yes — no approximation! |

where $B$ = block_size, $n$ = seq_len, $d$ = head_dim.

⚡ The memory savings come from never storing the full attention matrix — only one $B \times B$ tile at a time.

In [ ]:
from utils.attention_optimization import standard_attention, tiled_attention, flash_memory_analysis

# 💡 Demo: Flash attention matches standard attention exactly
mx.random.seed(42)
seq_len, d = 64, 32
Q = mx.random.normal((seq_len, d))
K = mx.random.normal((seq_len, d))
V = mx.random.normal((seq_len, d))
mx.eval(Q, K, V)

std_out = standard_attention(Q, K, V)
flash_out = tiled_attention(Q, K, V, block_size=16)
mx.eval(std_out, flash_out)

max_diff = float(mx.max(mx.abs(std_out - flash_out)))
print(f"Standard vs Flash attention (seq_len={seq_len}, d={d}, block_size=16):")
print(f"  Max absolute difference: {max_diff:.2e}")
print(f"  Match within 1e-5: {'\u2705 Yes' if max_diff < 1e-5 else '\u274c No'}")

# ⚡ Memory analysis
for sl in [512, 1024, 4096, 16384]:
    mem = flash_memory_analysis(sl, d=64, block_size=64)
    print(f"\n  seq_len={sl:>6d}: standard={mem['standard_total_bytes']/1e6:.1f}MB, "
          f"flash={mem['flash_total_bytes']/1e6:.1f}MB, "
          f"ratio={mem['memory_ratio']:.1f}x")

In [ ]:
# 🎯 Verify with different block sizes — all produce identical results
print("Flash attention correctness across block sizes:")
print(f"{'block_size':>10s}  {'max_diff':>10s}  {'status':>8s}")
print("-" * 32)
for bs in [4, 8, 16, 32, 64]:
    flash_bs = tiled_attention(Q, K, V, block_size=bs)
    mx.eval(flash_bs)
    diff = float(mx.max(mx.abs(std_out - flash_bs)))
    status = "✅" if diff < 1e-5 else "❌"
    print(f"{bs:>10d}  {diff:>10.2e}  {status:>8s}")
print(f"\n💡 Block size affects performance, not correctness — the algorithm is exact!")

## Paged Attention — Virtual Memory for KV-Cache

💡 **Analogy:** Just like an OS manages RAM with virtual memory pages, **Paged Attention** (vLLM, Kwon et al. 2023) manages KV-cache with fixed-size blocks.

### The Problem

During autoregressive generation, each request needs a KV-cache that grows with sequence length. With multiple concurrent requests:
- Pre-allocating max-length caches wastes memory (most sequences are shorter)
- Contiguous allocation causes **fragmentation**

### The Solution

| Concept | OS Virtual Memory | Paged Attention |
|---|---|---|
| **Unit** | Memory page (4KB) | KV block (e.g., 16 tokens) |
| **Allocation** | On demand | On demand |
| **Mapping** | Page table | Block table |
| **Benefit** | No fragmentation | No wasted KV-cache |

⚡ vLLM achieves 2–4x higher throughput than HuggingFace by eliminating KV-cache fragmentation.

In [ ]:
from utils.attention_optimization import PagedAttentionBlockManager

# 💡 Demo: Paged KV-cache block manager
mgr = PagedAttentionBlockManager(block_size=16, num_heads=4, head_dim=64, max_blocks=256)

# Simulate appending tokens from a generation request
mx.random.seed(42)
for i in range(50):
    k = mx.random.normal((4, 64))  # [num_heads, head_dim]
    v = mx.random.normal((4, 64))
    mx.eval(k, v)
    mgr.append_token(k, v)

stats = mgr.stats()
print("Paged KV-Cache Statistics:")
print(f"  Tokens stored:    {stats['num_tokens']}")
print(f"  Blocks allocated: {stats['num_active_blocks']}")
print(f"  Block size:       {stats['block_size']} tokens")
print(f"  Utilization:      {stats['utilization']:.1%}")
print(f"  Memory used:      {stats['memory_bytes'] / 1024:.1f} KB")

# Read back all KV pairs
keys, values = mgr.read_kv()
mx.eval(keys, values)
print(f"\n  Retrieved K shape: {keys.shape}")
print(f"  Retrieved V shape: {values.shape}")

# ⚡ Free some blocks
mgr.free_block(0)
print(f"\n  After freeing block 0: {mgr.num_active_blocks} active blocks")
print(f"\n💡 Blocks are allocated on demand and freed when done — no wasted memory!")

## Ring Attention — Distributed Sequence Parallelism

🎯 **Interview tip:** Ring Attention (Liu et al., 2024) enables context lengths of **1M+ tokens** by distributing the sequence across multiple devices in a ring topology.

### How It Works

1. **Split** Q, K, V into chunks — one per device
2. Each device holds its **local Q chunk** permanently
3. K/V chunks **circulate around the ring** (each device passes its KV to the next)
4. After $N$ steps (where $N$ = number of devices), every device has seen all K/V
5. Each device uses **online softmax** to combine partial results correctly

```
Device 0: Q_0 × [K_0, K_1, K_2, K_3]  (sees all K/V over 4 ring steps)
Device 1: Q_1 × [K_1, K_2, K_3, K_0]
Device 2: Q_2 × [K_2, K_3, K_0, K_1]
Device 3: Q_3 × [K_3, K_0, K_1, K_2]
```

⚡ Each device only needs memory for its local chunk + one remote KV chunk. Total memory scales as $O(n/P)$ per device where $P$ = number of devices.

In [ ]:
from utils.attention_optimization import simulate_ring

# 💡 Demo: Ring attention matches standard attention exactly
mx.random.seed(42)
seq_len, d = 64, 32
Q = mx.random.normal((seq_len, d))
K = mx.random.normal((seq_len, d))
V = mx.random.normal((seq_len, d))
mx.eval(Q, K, V)

std_out = standard_attention(Q, K, V)
mx.eval(std_out)

print("Ring Attention correctness across device counts:")
print(f"{'devices':>8s}  {'chunk_size':>10s}  {'max_diff':>10s}  {'status':>8s}")
print("-" * 42)
for n_dev in [2, 4, 8, 16]:
    ring_out = simulate_ring(Q, K, V, num_devices=n_dev)
    mx.eval(ring_out)
    diff = float(mx.max(mx.abs(std_out - ring_out)))
    chunk = seq_len // n_dev
    status = "✅" if diff < 1e-5 else "❌"
    print(f"{n_dev:>8d}  {chunk:>10d}  {diff:>10.2e}  {status:>8s}")

print(f"\n💡 Ring attention produces exact results regardless of device count!")
print(f"🎯 In production: each 'device' would be a separate GPU/TPU in a ring topology.")

## ⚡ Benchmarks: Standard vs Flash Attention

Let's measure wall-clock time at various sequence lengths to see where flash attention shines.

In [ ]:
import time
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from utils.attention_optimization import benchmark_attention

# ⚡ Benchmark at the required sequence lengths
seq_lengths = [512, 1024, 2048, 4096]
results = benchmark_attention(seq_lengths=seq_lengths, d=64, block_size=64, n_warmup=3, n_runs=5)

print("Benchmark: Standard vs Flash Attention")
print(f"{'seq_len':>8s}  {'standard':>10s}  {'flash':>10s}  {'mem_ratio':>10s}")
print("-" * 44)
for r in results:
    print(f"{r['seq_len']:>8d}  {r['standard_ms']:>8.2f}ms  {r['flash_ms']:>8.2f}ms  {r['memory_ratio']:>8.1f}x")

# Plot results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

slens = [r['seq_len'] for r in results]
std_times = [r['standard_ms'] for r in results]
flash_times = [r['flash_ms'] for r in results]
mem_ratios = [r['memory_ratio'] for r in results]

ax1.plot(slens, std_times, 'o-', label='Standard', color='#e74c3c', linewidth=2)
ax1.plot(slens, flash_times, 's-', label='Flash', color='#2ecc71', linewidth=2)
ax1.set_xlabel('Sequence Length')
ax1.set_ylabel('Time (ms)')
ax1.set_title('⚡ Attention Latency')
ax1.legend()
ax1.set_xscale('log', base=2)
ax1.grid(True, alpha=0.3)

ax2.bar(range(len(slens)), mem_ratios, color='#3498db', alpha=0.8)
ax2.set_xticks(range(len(slens)))
ax2.set_xticklabels([str(s) for s in slens])
ax2.set_xlabel('Sequence Length')
ax2.set_ylabel('Memory Savings (x)')
ax2.set_title('💡 Flash Attention Memory Savings')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('attention_benchmark.png', dpi=100, bbox_inches='tight')
plt.show()
print("\n⚡ Flash attention's advantage grows with sequence length!")

## 📜 History & Alternatives

The evolution of efficient attention mechanisms:

| Year | Innovation | Team | Key Contribution |
|------|-----------|------|------------------|
| 2017 | **Transformer** | Vaswani et al. (Google) | Original $O(n^2)$ scaled dot-product attention |
| 2019 | **Sparse Transformer** | Child et al. (OpenAI) | $O(n\sqrt{n})$ via sparse attention patterns |
| 2020 | **Linear Attention** | Katharopoulos et al. | $O(n)$ via kernel approximation (approximate) |
| 2022 | **Flash Attention** | Tri Dao et al. (Stanford) | IO-aware tiled attention — **exact** $O(n)$ memory, 2–4x faster. The key insight: compute is cheap, memory bandwidth is the bottleneck. |
| 2023 | **Flash Attention 2** | Tri Dao | Better work partitioning across GPU thread blocks, ~2x faster than FA1 |
| 2023 | **Paged Attention / vLLM** | Kwon et al. (UC Berkeley) | Virtual memory for KV-cache — eliminates fragmentation, 2–4x throughput |
| 2024 | **Ring Attention** | Liu et al. | Distributed sequence parallelism — enables 1M+ context by passing KV blocks around a device ring |
| 2024 | **Flash Attention 3** | Tri Dao et al. | Hopper GPU optimizations (FP8, warp specialization), approaching hardware limits |

### Key Takeaways

🎯 **For interviews:**
- Flash Attention is **exact** — it's not an approximation. It computes the same result as standard attention.
- The speedup comes from **reducing memory bandwidth**, not reducing computation.
- Online softmax is the mathematical trick that makes tiling possible.
- Paged Attention is orthogonal to Flash Attention — it optimizes **KV-cache management**, not attention computation.
- Ring Attention is for **multi-device** scenarios — on a single device, Flash Attention is sufficient.

⚡ **On Apple Silicon:** MLX's `mx.fast.scaled_dot_product_attention` uses Flash Attention under the hood. Our tiled implementation here is pedagogical — in production, use the built-in.

In [ ]:
# 💡 Summary: what we covered
print("═" * 60)
print("⚡ Notebook 12: Flash, Paged, and Ring Attention")
print("═" * 60)
print()
print("✅ Online Softmax: incremental softmax via running max/sum")
print("✅ Flash Attention: tiled attention, O(n) memory, exact")
print("✅ Paged Attention: virtual memory for KV-cache (vLLM)")
print("✅ Ring Attention: distributed attention across devices")
print("✅ Benchmarks: standard vs flash at various seq lengths")
print()
print("🎯 Key insight: Flash Attention is exact, not approximate.")
print("   The speedup comes from reducing memory bandwidth,")
print("   not reducing computation.")
print()
print("➡️  Next: Notebook 13 — Serving Locally")